# LAB 1 — LlamaIndex Parent-Child Retriever
## Aula 6: Indexação Avançada · MBA RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Objetivo:** Implementar o Parent-Child Retriever com LlamaIndex sobre corpus jurídico, compreendendo como separar unidade de busca (filho) de unidade de contexto (pai) melhora a qualidade.

**Tempo estimado:** 45 minutos | **Pré-requisito:** Aulas 1-5 concluídas

### Mapa do Lab:
1. Instalação e setup
2. Carregar corpus jurídico
3. Configurar HierarchicalNodeParser
4. Criar índice e AutoMergingRetriever
5. Comparar com chunking plano
6. Salvar métricas para Lab 4

## ⚙️ Passo 1 — Instalação

*Tempo estimado: 3-5 min*

In [ ]:
!pip install llama-index llama-index-core llama-index-embeddings-huggingface --quiet
!pip install sentence-transformers faiss-cpu langchain-openai ragas --quiet
print('✅ Dependências instaladas')

## 📦 Passo 2 — Imports e Verificação

In [ ]:
import sys, json, os, warnings
import numpy as np
warnings.filterwarnings('ignore')
assert sys.version_info >= (3, 11), 'Use Python 3.11+'

from llama_index.core import Document, StorageContext
from llama_index.core.node_parser import (
    HierarchicalNodeParser,
    SentenceSplitter,
    get_leaf_nodes,
    get_root_nodes,
)
from llama_index.core import VectorStoreIndex
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print('✅ Imports realizados')
print(f'Python: {sys.version[:6]}')

## 📚 Passo 3 — Carregar o Corpus Jurídico

In [ ]:
# Tentar carregar corpus do arquivo; fallback inline
try:
    with open('corpus_indexacao_avancada.json', 'r', encoding='utf-8') as f:
        corpus_data = json.load(f)
    docs_raw = corpus_data['documentos']
    queries_bench = corpus_data['perguntas_teste']
    print(f'✅ Corpus carregado: {len(docs_raw)} documentos')
except FileNotFoundError:
    print('Arquivo nao encontrado. Usando corpus inline...')
    docs_raw = [
        {'id':'DOC001','tipo':'acordao','numero':'HC 143641/SP',
         'texto':'O STF determinou a substituicao da prisao preventiva pela domiciliar para mulheres gestantes ou maes de criancas ate 12 anos. A Lei 13257/2016 alterou o art 318 do CPP. O sistema penitenciario nao tem estrutura para atender maes que amamentam nem para garantir cuidados minimos as criancas. Os efeitos sao erga omnes aplicando-se a todo o territorio nacional.'},
        {'id':'DOC002','tipo':'legislacao','numero':'Lei 13.964/2019',
         'texto':'O Pacote Anticrime criou o juiz das garantias responsavel pelo controle da legalidade da investigacao criminal e pela salvaguarda dos direitos individuais. O juiz das garantias e competente para receber a comunicacao de prisao, receber o auto de prisao em flagrante e zelar pelos direitos do preso. A cadeia de custodia e o conjunto de procedimentos para manter e documentar a historia cronologica do vestigio coletado.'},
        {'id':'DOC006','tipo':'acordao','numero':'RHC 143.308/SP',
         'texto':'Para busca pessoal sem mandado judicial e necessaria fundada suspeita objetiva e concreta de que o individuo porta objeto ilicito ou arma proibida conforme art 240 paragrafo 2 CPP. A simples presenca em local de trafico e atitudes nervosas nao constituem fundada suspeita. Provas obtidas mediante busca pessoal ilegal sao ilicitas teoria dos frutos da arvore envenenada. RECURSO PROVIDO para trancar a acao penal.'},
    ]
    queries_bench = [
        {'id':'Q001','pergunta':'Quais os criterios para substituicao de prisao por domiciliar para maes?','resposta_esperada':'Gestantes, puerp. ou maes de criancas ate 12 anos'},
        {'id':'Q003','pergunta':'Quais os requisitos para busca pessoal sem mandado?','resposta_esperada':'Fundada suspeita objetiva e concreta'},
    ]
    print(f'✅ Corpus inline: {len(docs_raw)} documentos')

# Converter para Document do LlamaIndex
documents = [
    Document(text=d['texto'], metadata={'id':d['id'],'numero':d.get('numero','')})
    for d in docs_raw
]
print(f'✅ {len(documents)} documentos LlamaIndex criados')

## 🏗️ Passo 4 — HierarchicalNodeParser

O parser cria dois níveis:
- **Pai (512 tokens):** contexto completo para o LLM
- **Filho (128 tokens):** unidade de busca precisa

*Ratio 1:4 — cada pai contém ~4 filhos*

In [ ]:
# Configurar parser hierarquico
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[512, 128],
    chunk_overlap=20,
)

print('Parseando documentos...')
all_nodes = node_parser.get_nodes_from_documents(documents, show_progress=True)
leaf_nodes = get_leaf_nodes(all_nodes)
root_nodes = get_root_nodes(all_nodes)

print(f'Total nos: {len(all_nodes)}')
print(f'Folha (filhos): {len(leaf_nodes)}')
print(f'Raiz (pais): {len(root_nodes)}')
print(f'Ratio medio: {len(leaf_nodes)/max(len(root_nodes),1):.1f} filhos/pai')

# Inspecionar par pai-filho
if leaf_nodes:
    filho = leaf_nodes[0]
    print(f'\nFILHO ({len(filho.text)} chars): {filho.text[:150]}')
    pai = next((n for n in all_nodes if n.node_id == filho.parent_node.node_id), None) if filho.parent_node else None
    if pai:
        print(f'PAI ({len(pai.text)} chars): {pai.text[:250]}')

## 📊 Passo 5 — Criar Índice e AutoMergingRetriever

In [ ]:
# Modelo de embedding
try:
    embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-m3', max_length=512)
    print('BGE-M3 carregado')
except Exception as e:
    print(f'Fallback: {e}')
    embed_model = HuggingFaceEmbedding(model_name='sentence-transformers/all-MiniLM-L6-v2')
    print('all-MiniLM-L6-v2 carregado')

# Docstore armazena todos os nos (pais e filhos)
docstore = SimpleDocumentStore()
docstore.add_documents(all_nodes)
storage_context = StorageContext.from_defaults(docstore=docstore)

# Indexar SOMENTE os filhos no vector store
print('Criando indice vetorial com nos folha...')
base_index = VectorStoreIndex(
    leaf_nodes, storage_context=storage_context,
    embed_model=embed_model, show_progress=True,
)

# AutoMergingRetriever: se >= 30% filhos de um pai forem recuperados,
# substitui pelos filhos pelo pai inteiro
base_retriever = base_index.as_retriever(similarity_top_k=6)
retriever = AutoMergingRetriever(
    base_retriever, storage_context,
    verbose=True, simple_ratio_thresh=0.3,
)
print('AutoMergingRetriever configurado (threshold 30%)')

## 🔬 Passo 6 — Testar e Comparar com Chunking Plano

In [ ]:
# Indexacao plana (baseline desta aula)
flat_parser = SentenceSplitter(chunk_size=256, chunk_overlap=20)
flat_nodes = flat_parser.get_nodes_from_documents(documents)
flat_index = VectorStoreIndex(flat_nodes, embed_model=embed_model, show_progress=False)
flat_retriever = flat_index.as_retriever(similarity_top_k=4)

queries = [q['pergunta'] for q in queries_bench[:3]]
if not queries:
    queries = ['Quais os requisitos para busca pessoal sem mandado judicial?']

print('COMPARACAO: Flat vs Parent-Child')
print('='*60)

pc_scores, flat_scores = [], []

for query in queries:
    flat_res = flat_retriever.retrieve(query)
    pc_res = retriever.retrieve(query)

    flat_chars = sum(len(n.text) for n in flat_res)
    pc_chars = sum(len(n.text) for n in pc_res)

    flat_avg = np.mean([n.score for n in flat_res if n.score]) if flat_res else 0
    pc_avg = np.mean([n.score for n in pc_res if n.score]) if pc_res else 0

    flat_scores.append(flat_avg)
    pc_scores.append(pc_avg)

    print(f'Query: {query[:50]}...')
    print(f'  Flat:         {len(flat_res)} nos | {flat_chars} chars | score {flat_avg:.3f}')
    print(f'  Parent-Child: {len(pc_res)} nos | {pc_chars} chars | score {pc_avg:.3f}')
    print()

print(f'Score medio Flat: {np.mean(flat_scores):.3f}')
print(f'Score medio Parent-Child: {np.mean(pc_scores):.3f}')
print(f'Melhoria: {np.mean(pc_scores)-np.mean(flat_scores):+.3f}')

assert len(pc_res) > 0, 'Parent-Child nao retornou resultados'
print('Checkpoint: OK')

## 🏋️ Exercício — Experimento com Thresholds

In [ ]:
# 🏋️ EXERCICIO: Ajuste o threshold e observe o impacto
print('Experimento: Impacto do threshold de merging')
print('='*55)

query_exp = queries[0] if queries else 'busca pessoal fundada suspeita'

for threshold in [0.1, 0.3, 0.5, 0.8]:
    ret_exp = AutoMergingRetriever(
        base_retriever, storage_context,
        verbose=False, simple_ratio_thresh=threshold,
    )
    res = ret_exp.retrieve(query_exp)
    chars = sum(len(n.text) for n in res)
    pais = sum(1 for n in res if len(n.text) > 200)
    filhos = len(res) - pais
    print(f'Threshold {threshold:.1f}: {len(res)} nos ({pais} pais + {filhos} filhos) | {chars} chars ao LLM')

print('\nRecomendacao para juridico: threshold 0.3-0.4 (equilibrio precisao/contexto)')

## 💾 Passo 7 — Salvar Resultados para Lab 4

In [ ]:
import os
os.makedirs('resultados_aula6', exist_ok=True)

resultados_pc = {
    'tecnica': 'Parent-Child',
    'config': {'chunk_pai': 512, 'chunk_filho': 128, 'threshold': 0.3},
    'total_nos_filho': len(leaf_nodes),
    'total_nos_pai': len(root_nodes),
    'score_medio': float(np.mean(pc_scores)) if pc_scores else 0.0,
    'queries_testadas': queries,
}

with open('resultados_aula6/resultados_parent_child.json', 'w', encoding='utf-8') as f:
    json.dump(resultados_pc, f, ensure_ascii=False, indent=2)

print('✅ Salvo: resultados_aula6/resultados_parent_child.json')
print(json.dumps(resultados_pc, indent=2, ensure_ascii=False))

## 📚 Referências (ABNT)

LLAMAINDEX. **Auto-Merging Retriever**. LlamaIndex Documentation, 2024. Disponível em: <https://docs.llamaindex.ai>. Acesso em: abr. 2026.

---
*MBA em RAG & CAG Aplicados a Direito e Segurança Pública — Aula 6, Lab 1*